# E2 · Reglas posteriores a la predicción
**Parte de:** E0.

**Hipótesis.** Tres reglas deterministas podrían corregir errores:
1. **Comillas → AG = 1.** En el train, las 39 filas entrecomilladas tienen AG = 1 (H10). En el test hay 25.
2. **Duplicados.** El test tiene 5 textos repetidos y deberían recibir la misma predicción.
3. **Filas `#¿NOMBRE?`.** El test tiene 6 filas sin texto recuperable que hoy reciben `000`. ¿Hay algo mejor?

### Glosario de esta prueba
| Término | Significado |
|---|---|
| **HS / TR / AG** | Las 3 etiquetas: discurso de odio / odio dirigido a una persona concreta / agresividad |
| **F1 medio** | F1 de la clase 1 de cada etiqueta, promediado entre HS, TR y AG (métrica principal) |
| **CV repetida** | Validación cruzada de 5 particiones repetida con 3 semillas (42, 43, 44); se reporta media ± desviación |
| **OOF** | *Out-of-fold*: predicción de cada fila del train hecha por un modelo que no la vio al entrenar |
| **Stacking** | Un metamodelo (regresión logística) aprende a combinar las probabilidades OOF de los modelos base |
| **Regla posterior** | Ajuste determinista aplicado a las predicciones del modelo, después de binarizar |
| **`#¿NOMBRE?`** | Error de Excel que reemplazó el texto de algunas celdas |

El glosario completo está en `RESUMEN.md`.

In [1]:
import os, sys, platform
sys.path.insert(0, os.path.abspath('.'))          # el notebook vive en experimentos/, junto a toxic_base.py
import numpy as np, pandas as pd, sklearn, scipy
import toxic_base as tb
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30); pd.set_option('display.max_colwidth', 120)
print('Python', platform.python_version(), '| numpy', np.__version__, '| pandas', pd.__version__,
      '| scikit-learn', sklearn.__version__, '| scipy', scipy.__version__)
print('Semillas de la CV repetida:', tb.SEEDS, '| particiones:', tb.N_FOLDS)
d = tb.Datos()
print('Datos en', d.data_dir, '| train', d.train.shape, '| test', d.test.shape)

Python 3.12.3 | numpy 2.5.4 | pandas 3.0.6 | scikit-learn 1.9.1 | scipy 1.18.1
Semillas de la CV repetida: (42, 43, 44) | particiones: 5


Datos en /home/jc/repositories/natural-language-processing-workshops/workshops/Kaggle-tweet-toxic/tweet-toxic-2026-puj | train (1731, 5) | test (1154, 2)


## Regla 1 · comillas → AG = 1

In [2]:
com_tr = d.train['text'].str.contains('"', regex=False).values
def regla_comillas(Yp):
    Yp = Yp.copy(); Yp[com_tr, 2] = 1; return Yp

ST = {s: tb.stacking(d, tb.BASES_ORIGINALES, s, con_test=False)[0] for s in tb.SEEDS}
print('Proporción de filas entrecomilladas del train con AG predicho = 1 (OOF):',
      [round(tb.binarizar(P)[com_tr, 2].mean(), 3) for P in ST.values()])
tb.tabla({'E0 · stacking': tb.evaluar(d, ST), '+ comillas → AG=1': tb.evaluar(d, ST, post=regla_comillas)})

Proporción de filas entrecomilladas del train con AG predicho = 1 (OOF): [np.float64(1.0), np.float64(1.0), np.float64(1.0)]


,F1 HS,F1 TR,F1 AG,F1 medio,±,EMR,F1 medio por semilla
Configuración,,,,,,,
E0 · stacking,0.8038,0.8345,0.7994,0.8126,0.0021,0.7757,0.815 / 0.810 / 0.813
+ comillas → AG=1,0.8038,0.8345,0.7994,0.8126,0.0021,0.7757,0.815 / 0.810 / 0.813


In [3]:
_, Pt = tb.stacking(d, tb.BASES_ORIGINALES, 42)
Yt = tb.binarizar(Pt)
print('Filas entrecomilladas en test:', d.comillas_te.sum(), '| con AG predicho = 1:', Yt[d.comillas_te, 2].sum())

Filas entrecomilladas en test: 25 | con AG predicho = 1: 25


## Regla 2 · duplicados del test

In [4]:
dup = d.test[d.test['text'].duplicated(keep=False)].copy()
dup[tb.LABELS] = Yt[dup.index]
dup.sort_values('text')[['id', 'text'] + tb.LABELS]

,id,text,HS,TR,AG
168,22557,#¿NOMBRE?,0,0,0
576,20477,#¿NOMBRE?,0,0,0
610,24998,#¿NOMBRE?,0,0,0
736,20081,#¿NOMBRE?,0,0,0
737,22614,#¿NOMBRE?,0,0,0
761,24635,#¿NOMBRE?,0,0,0


## Regla 3 · filas `#¿NOMBRE?`

In [5]:
err_tr = d.train['text'].str.strip().str.match(tb.RE_ERROR_EXCEL)
print('En train:', err_tr.sum(), 'fila(s):'); display(d.train.loc[err_tr, ['id', 'text'] + tb.LABELS])
print('Combinación más frecuente en train:', pd.Series(d.COMBO).value_counts().index[0])
print('Filas #¿NOMBRE? en test:', d.err_excel_te.sum())

En train: 1 fila(s):


,id,text,HS,TR,AG
1031,21055,#NOME?,0,0,0


Combinación más frecuente en train: 000
Filas #¿NOMBRE? en test: 6


## Conclusión: **no se adopta ninguna regla** (no cambian nada)
- **Comillas:** el modelo ya predice AG = 1 en el 100 % de las filas entrecomilladas, tanto en el train (OOF) como en el test, gracias a la variable `LEX:comillas`. La regla no cambia ninguna predicción.
- **Duplicados:** los 5 textos repetidos del test son justamente las filas `#¿NOMBRE?`, así que ya reciben la misma predicción (`000`).
- **`#¿NOMBRE?`:** solo hay un ejemplo en el train y la combinación más frecuente es `000`, que es lo que ya se asigna.
- Resultado negativo pero útil para el informe: confirma que el modelo ya aprendió el artefacto de las comillas. No se genera submission.